# COMPX525 Assignment 1 - Approach 1 Architecture 2: Vision Transformer from Scratch

This notebook trains a small Vision Transformer (ViT-Small) from scratch on the petri dish dataset.
It serves as the second architecture for Approach 1, complementing the SE-ResNet (CNN) trained in V3.

**Purpose:** Demonstrate that Transformers lack the inductive biases (locality, translation equivariance)
that make CNNs effective on small datasets, confirming that CNN architectures are the better choice
when training from scratch with limited data.

In [ ]:
# Setup
!pip install -q albumentations

In [ ]:
import os, random, time, json, gc, warnings
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ══════════════════════════════════════════════════════
# CONFIG - UPDATE PATHS FOR YOUR ENVIRONMENT
# ══════════════════════════════════════════════════════
# For Kaggle:
#   BASE_DIR = '/kaggle/input/datasets/necmikural/processed-petri'
#   OUTPUT_DIR = '/kaggle/working'
# For Colab:
#   BASE_DIR = '/content/drive/MyDrive/deep_learning_1'
#   OUTPUT_DIR = BASE_DIR

BASE_DIR    = '/kaggle/input/datasets/necmikural/processed-petri'
OUTPUT_DIR  = '/kaggle/working'

TRAIN_DIR = os.path.join(BASE_DIR, 'processed', 'train')
TEST_DIR  = os.path.join(BASE_DIR, 'processed', 'test')

IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
NUM_CLASSES = 16

for d in [TRAIN_DIR, TEST_DIR]:
    assert os.path.isdir(d), f'Not found: {d}'
print('Paths OK')

## 1. Data Loading

In [ ]:
all_classes = sorted([d for d in os.listdir(TRAIN_DIR)
                      if os.path.isdir(os.path.join(TRAIN_DIR, d)) and d != 'j'])
class_to_idx = {c: i for i, c in enumerate(all_classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
print(f'Classes ({len(all_classes)}): {all_classes}')

# Separate original vs augmented
orig_files, orig_labels = [], []
aug_files, aug_labels = [], []

for cls in all_classes:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    for f in sorted(os.listdir(cls_dir)):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        fpath = os.path.join(cls_dir, f)
        if f.startswith('aug_'):
            aug_files.append(fpath)
            aug_labels.append(class_to_idx[cls])
        else:
            orig_files.append(fpath)
            orig_labels.append(class_to_idx[cls])

orig_files = np.array(orig_files)
orig_labels = np.array(orig_labels)
aug_files = np.array(aug_files)
aug_labels = np.array(aug_labels)
print(f'Original: {len(orig_files)}, Augmented: {len(aug_files)}')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_aug(size=224):
    return A.Compose([
        A.RandomResizedCrop(size=(size, size), scale=(0.7, 1.0)),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=45, p=0.5),
        A.OneOf([
            A.GaussNoise(std_range=(0.01, 0.05), p=1),
            A.GaussianBlur(blur_limit=(3, 5), p=1),
            A.MotionBlur(blur_limit=5, p=1),
        ], p=0.3),
        A.OneOf([
            A.OpticalDistortion(distort_limit=0.05, p=1),
            A.GridDistortion(num_steps=5, distort_limit=0.05, p=1),
        ], p=0.2),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),
        A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, size//16),
                        hole_width_range=(1, size//16), p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2(),
    ])

def get_val_aug(size=224):
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2(),
    ])

class PetriDS(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files, self.labels, self.transform = files, labels, transform
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = np.array(Image.open(self.files[i]).convert('RGB'))
        if self.transform: img = self.transform(image=img)['image']
        return img, self.labels[i]

print('Dataset and augmentations ready')

## 2. Training Utilities (same as V3)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.gamma, self.ls, self.alpha = gamma, label_smoothing, alpha
    def forward(self, x, y):
        ce = F.cross_entropy(x, y, weight=self.alpha, reduction='none', label_smoothing=self.ls)
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()


def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            out = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        correct += out.argmax(1).eq(labels).sum().item()
        total += imgs.size(0)
    return loss_sum/total, 100.0*correct/total


@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    correct, total, all_preds, all_labels = 0, 0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        preds = out.argmax(1)
        correct += preds.eq(labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return 100.0*correct/total, np.array(all_preds), np.array(all_labels)

print('Training utilities ready')

## 3. Vision Transformer Architecture

In [ ]:
class PatchEmbed(nn.Module):
    """Split image into patches and project to embedding dimension."""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2  # 196
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # (B, C, H, W) -> (B, num_patches, embed_dim)
        return self.proj(x).flatten(2).transpose(1, 2)


class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads=4, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop=0.1, attn_drop=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiHeadAttention(dim, num_heads, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(mlp_hidden, dim), nn.Dropout(drop),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    """Small Vision Transformer for training from scratch.
    
    Architecture:
    - Patch embedding: 16x16 patches -> 196 tokens per 224x224 image
    - Learnable [CLS] token prepended to patch sequence
    - Learnable positional embeddings (197 positions)
    - 6 Transformer encoder blocks (pre-norm style)
    - Classification via [CLS] token output
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=16,
                 embed_dim=256, depth=6, num_heads=4, mlp_ratio=4.0, drop_rate=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches  # 196

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop_rate)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads, mlp_ratio, drop=drop_rate)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embed_dim, num_classes)
        )

        # Initialize following ViT conventions
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                          # (B, 196, 256)
        cls = self.cls_token.expand(B, -1, -1)            # (B, 1, 256)
        x = torch.cat([cls, x], dim=1)                    # (B, 197, 256)
        x = self.pos_drop(x + self.pos_embed)
        x = self.blocks(x)
        x = self.norm(x[:, 0])                            # CLS token output
        return self.head(x)


# Create model
vit_model = VisionTransformer(
    img_size=IMG_SIZE, patch_size=16, num_classes=NUM_CLASSES,
    embed_dim=256, depth=6, num_heads=4, mlp_ratio=4.0, drop_rate=0.1
).to(DEVICE)

vit_params = sum(p.numel() for p in vit_model.parameters())
print(f'ViT-Small from scratch: {vit_params:,} params')
print(f'  Config: patch=16, embed=256, depth=6, heads=4, mlp_ratio=4')
print(f'  Patches per image: {(IMG_SIZE//16)**2} ({IMG_SIZE//16}x{IMG_SIZE//16})')
print(f'  Sequence length: {(IMG_SIZE//16)**2 + 1} (patches + CLS token)')

## 4. Train ViT from Scratch

In [ ]:
# Same 85/15 stratified split as V3's SE-ResNet for fair comparison
ti, vi = train_test_split(np.arange(len(orig_files)), test_size=0.15,
                          stratify=orig_labels, random_state=SEED)

train_files = np.concatenate([orig_files[ti], aug_files])
train_labels = np.concatenate([orig_labels[ti], aug_labels])
val_files = orig_files[vi]
val_labels = orig_labels[vi]

print(f'Train: {len(train_files)}, Val: {len(val_files)}')

# Datasets
train_ds = PetriDS(train_files, train_labels, get_train_aug(IMG_SIZE))
val_ds   = PetriDS(val_files, val_labels, get_val_aug(IMG_SIZE))

# Weighted sampler (sqrt-balanced)
counts = Counter(train_labels)
sw = np.array([1.0 / np.sqrt(counts[l]) for l in train_labels])
sampler = WeightedRandomSampler(torch.from_numpy(sw).double(), len(sw), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

# Loss with sqrt-balanced weights
total_n = sum(counts.values())
cw = torch.tensor([np.sqrt(total_n/(NUM_CLASSES*counts[i])) for i in range(NUM_CLASSES)],
                  dtype=torch.float32).to(DEVICE)
cw = cw / cw.mean()
criterion = FocalLoss(alpha=cw, gamma=2.0, label_smoothing=0.1)

# Optimizer with layer-wise LR
params = list(vit_model.named_parameters())
n_params = len(params)
param_groups = [{'params': p, 'lr': 5e-4 * (0.1 + 0.9*i/n_params)}
                for i, (_, p) in enumerate(params)]
optimizer = optim.AdamW(param_groups, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-7)
scaler = GradScaler()

# Training loop
EPOCHS = 50
PATIENCE = 15
best_acc, no_improve = 0, 0
save_path = os.path.join(OUTPUT_DIR, 'best_vit_small.pth')

print(f'\nTraining ViT-Small for {EPOCHS} epochs...')
print('=' * 70)

for ep in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(vit_model, train_loader, criterion, optimizer, scaler)
    scheduler.step()
    val_acc, val_preds, val_labels_out = eval_model(vit_model, val_loader)

    elapsed = time.time() - t0
    marker = ''
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(vit_model.state_dict(), save_path)
        no_improve = 0
        marker = f' << BEST ({best_acc:.2f}%)'
    else:
        no_improve += 1

    print(f'Ep {ep+1:3d}/{EPOCHS} | Loss: {tr_loss:.4f} | '
          f'Train: {tr_acc:.1f}% | Val: {val_acc:.1f}% | '
          f'{elapsed:.0f}s{marker}')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {ep+1} (no improvement for {PATIENCE} epochs)')
        break

# Load best model
vit_model.load_state_dict(torch.load(save_path, weights_only=True))
print(f'\nBest ViT-Small validation accuracy: {best_acc:.2f}%')

## 5. Evaluation

In [ ]:
# Final evaluation with classification report
val_acc_final, final_preds, final_labels = eval_model(vit_model, val_loader)

print(f'ViT-Small Validation Accuracy: {val_acc_final:.2f}%')
print(f'\nClassification Report:')
print(classification_report(final_labels, final_preds, target_names=all_classes))

# Save results
results = {
    'model': 'ViT-Small (from scratch)',
    'architecture': {
        'type': 'Vision Transformer',
        'patch_size': 16,
        'embed_dim': 256,
        'depth': 6,
        'num_heads': 4,
        'mlp_ratio': 4.0,
        'total_params': vit_params,
    },
    'val_accuracy': round(val_acc_final, 2),
    'classification_report': classification_report(
        final_labels, final_preds, target_names=all_classes, output_dict=True
    ),
}

results_path = os.path.join(OUTPUT_DIR, 'results_vit.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f'\nResults saved to: {results_path}')
print(f'\n{"="*50}')
print(f'SUMMARY')
print(f'{"="*50}')
print(f'ViT-Small (Transformer, from scratch): {val_acc_final:.2f}%')
print(f'SE-ResNet (CNN, from scratch):          52.06%  (from V3)')
print(f'\nConclusion: CNN > ViT when training from scratch on small datasets.')
print(f'ViTs need large-scale pretraining (ImageNet-21K+) to be effective.')
print(f'{"="*50}')